## Mapping MIMIC_IV Patients' Drug Allergies to RxNORM
This notebook
1. normalizes entries in MIMIC_IV patients' drug allergy entries (which was extracted from MIMIC_IV_EXT_BHC notes) to RxNORM


In [ ]:
# Configuration
from dotenv import load_dotenv
import os
import requests
import time
import pandas as pd
from difflib import get_close_matches
import re
import json
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

load_dotenv()
UMLS_API_KEY = os.getenv('UMLS_API_KEY')

if not UMLS_API_KEY:
    print("Warning: UMLS_API_KEY not found in .env — SNOMED lookup will be skipped")
else:
    print(f"UMLS key loaded: {UMLS_API_KEY[:2]}...{UMLS_API_KEY[-2:]}")

RXNORM_BASE = "https://rxnav.nlm.nih.gov/REST"
RXCLASS_BASE = "https://rxnav.nlm.nih.gov/REST/rxclass"
UMLS_CONTENT_URL = "https://uts-ws.nlm.nih.gov/rest/content/current"
UMLS_AUTH_URL  = "https://utslogin.nlm.nih.gov/cas/v1/api-key"
UMLS_SEARCH_URL = "https://uts-ws.nlm.nih.gov/rest/search/current"
UMLS_CROSSWALK_URL = "https://uts-ws.nlm.nih.gov/rest/crosswalk/current/source/RXNORM"


In [ ]:
# ── 1. Text cleaning ──────────────────────────────────────────────────────────

def clean_drug_name(name):
    """Basic cleanup before querying."""
    name = str(name).strip()
    name = re.sub(r'\(.*?\)', '', name)   # remove parenthetical notes
    name = re.sub(r'\s+', ' ', name)      # collapse whitespace
    return name.strip()

In [ ]:
# ── 1. Requests session with automatic retries ────────────────────────────────

def make_session():
    """
    Retry on connection errors and 5xx responses.
    Exponential backoff: 1s, 2s, 4s between retries.
    """
    session = requests.Session()
    retry = Retry(
        total=5,                            # max retry attempts
        backoff_factor=1,                   # 1s, 2s, 4s, 8s, 16s
        status_forcelist=[429, 500, 502, 503, 504],
        allowed_methods=["GET", "POST"],
        raise_on_status=False
    )
    adapter = HTTPAdapter(max_retries=retry)
    session.mount("https://", adapter)
    session.mount("http://", adapter)
    return session

SESSION = make_session()

In [ ]:
# ── 2. Wrap every API call with an extra safety net ───────────────────────────

def safe_get(url, params=None, headers=None, max_attempts=5):
    """
    Catches ConnectionError / timeout separately from HTTP errors.
    Falls back gracefully instead of crashing the whole run.
    """
    for attempt in range(max_attempts):
        try:
            r = SESSION.get(url, params=params, headers=headers, timeout=15)
            r.raise_for_status()
            return r
        except (requests.ConnectionError, requests.Timeout) as e:
            wait = 2 ** attempt          # 1, 2, 4, 8, 16 seconds
            print(f"  Connection error (attempt {attempt+1}/{max_attempts}), "
                  f"retrying in {wait}s: {e}")
            time.sleep(wait)
        except requests.HTTPError as e:
            print(f"  HTTP error: {e}")
            return None
    print(f"  Giving up on {url}")
    return None

In [ ]:
# ── 3. RxNorm lookup (exact → approximate → spelling) ────────────────────────

def get_rxcui_exact(drug_name):
    r = safe_get(f"{RXNORM_BASE}/rxcui.json",
                 params={"name": drug_name, "search": 0})
    if r is None:
        return None
    try:
        return r.json()['idGroup']['rxnormId'][0]
    except (KeyError, IndexError):
        return None

def get_rxcui_approx(drug_name):
    r = safe_get(f"{RXNORM_BASE}/rxcui.json",
                 params={"name": drug_name, "search": 2})
    if r is None:
        return None
    try:
        return r.json()['idGroup']['rxnormId'][0]
    except (KeyError, IndexError):
        return None

def get_rxcui_spelling(drug_name):
    r = safe_get(f"{RXNORM_BASE}/spellingsuggestions.json",
                 params={"name": drug_name})
    if r is None:
        return None, None
    try:
        suggestions = r.json()['suggestionGroup']['suggestionList']['suggestion']
        if suggestions:
            rxcui = get_rxcui_exact(suggestions[0])
            return rxcui, suggestions[0]
    except (KeyError, TypeError):
        pass
    return None, None

def get_rxcui(drug_name):
    cleaned = clean_drug_name(drug_name)

    rxcui = get_rxcui_exact(cleaned)
    if rxcui:
        return rxcui, cleaned, 'exact'

    rxcui = get_rxcui_approx(cleaned)
    if rxcui:
        return rxcui, cleaned, 'approx'

    rxcui, suggested = get_rxcui_spelling(cleaned)
    if rxcui:
        return rxcui, suggested, 'spelling'

    return None, None, 'no_match'

In [ ]:
# ── 4. Checkpoint: save progress to disk as you go ───────────────────────────

CHECKPOINT_FILE = "drug_lookup_checkpoint.json"

def load_checkpoint():
    """Load previously completed results."""
    if os.path.exists(CHECKPOINT_FILE):
        with open(CHECKPOINT_FILE, 'r') as f:
            data = json.load(f)
        print(f"Resuming from checkpoint: {len(data)} entries already done.")
        return data
    return {}

def save_checkpoint(results_dict):
    """Save current results dict to disk."""
    with open(CHECKPOINT_FILE, 'w') as f:
        json.dump(results_dict, f)

In [ ]:
# ── 3. Concept metadata ───────────────────────────────────────────────────────

def get_concept_info(rxcui):
    r = safe_get(f"{RXNORM_BASE}/rxcui/{rxcui}/properties.json")
    if r is None:
        return {}
    try:
        props = r.json()['properties']
        return {'canonical_name': props.get('name'), 'tty': props.get('tty')}
    except (KeyError, TypeError):
        return {}

In [ ]:
# ── 4. RxClass hierarchy for a known rxcui ───────────────────────────────────

def get_classes(rxcui, rela_source='ATC'):
    r = safe_get(f"{RXCLASS_BASE}/class/byRxcui.json",
                 params={"rxcui": rxcui, "relaSource": rela_source})
    if r is None:
        return []
    classes = []
    try:
        for entry in r.json()['rxclassDrugInfoList']['rxclassDrugInfo']:
            c = entry['rxclassMinConceptItem']
            classes.append({
                'class_id':   c['classId'],
                'class_name': c['className'],
                'class_type': c['classType'],
                'rela':       entry['rela'],
                'source':     rela_source
            })
    except (KeyError, TypeError):
        pass
    return classes

def get_atc_hierarchy(rxcui):
    """Try ATC first, fall back to MEDRT then FDASPL."""
    classes = []
    used_source = None
    for src in ['ATC', 'MEDRT', 'FDASPL']:
        classes = get_classes(rxcui, src)
        if classes:
            used_source = src
            break

    if not classes:
        return {'level_1_class': None, 'level_1_id': None,
                'level_2_class': None, 'level_2_id': None,
                'class_source':  None}

    # ATC codes: 3-char = L2 (our level_1), 5-char = L3 (our level_2)
    atc_by_len = {}
    others = []
    for c in classes:
        cid = c['class_id']
        if re.match(r'^[A-Z]\d{2}', cid):
            atc_by_len[len(cid)] = c
        else:
            others.append(c)

    if atc_by_len:
        l1 = atc_by_len.get(3) or atc_by_len.get(4)
        l2 = atc_by_len.get(5) or atc_by_len.get(4)
        return {
            'level_1_class': l1['class_name'] if l1 else None,
            'level_1_id':    l1['class_id']   if l1 else None,
            'level_2_class': l2['class_name'] if l2 else None,
            'level_2_id':    l2['class_id']   if l2 else None,
            'class_source':  'ATC'
        }
    else:
        unique = list({c['class_name']: c for c in others}.values())
        return {
            'level_1_class': unique[0]['class_name'] if len(unique) > 0 else None,
            'level_1_id':    unique[0]['class_id']   if len(unique) > 0 else None,
            'level_2_class': unique[1]['class_name'] if len(unique) > 1 else None,
            'level_2_id':    unique[1]['class_id']   if len(unique) > 1 else None,
            'class_source':  used_source
        }

In [ ]:
def get_snomed_from_rxcui(rxcui):
    """Crosswalk RXNORM RXCUI → SNOMED CT code via UMLS crosswalk endpoint."""
    r = safe_get(
        f"{UMLS_CROSSWALK_URL}/{rxcui}",
        params={
            "targetSource": "SNOMEDCT_US",
            "apiKey": UMLS_API_KEY
        }
    )
    if r is None:
        return None
    try:
        results = r.json()['result']
        snomed_codes = [item['ui'] for item in results 
                        if item.get('rootSource') == 'SNOMEDCT_US'
                        and item.get('ui') != 'NONE']
        return snomed_codes[0] if snomed_codes else None
    except (KeyError, IndexError, TypeError):
        return None

In [ ]:
# ── 5. RxClass direct lookup for category-style entries ─────────────────

def lookup_as_class(name):
    cleaned = clean_drug_name(name)
    for src in ['ATC', 'MEDRT', 'FDASPL', 'MESH']:
        r = safe_get(f"{RXCLASS_BASE}/class/byName.json",
                     params={"className": cleaned, "relaSource": src})
        if r is None:
            continue
        try:
            concepts = r.json()['rxclassMinConceptList']['rxclassMinConcept']
            if concepts:
                c = concepts[0]
                return c['classId'], c['className'], c['classType'], src
        except (KeyError, IndexError, TypeError):
            continue
    return None, None, None, None

def get_class_parent(class_id, source='ATC'):
    r = safe_get(f"{RXCLASS_BASE}/classTree.json",
                 params={"classId": class_id, "relaSource": source})
    if r is None:
        return None, None
    try:
        tree = r.json()['rxclassTree']['rxclassMinConcept']
        if len(tree) > 1:
            parent = tree[1]
            return parent['classId'], parent['className']
    except (KeyError, TypeError, IndexError):
        pass
    return None, None

In [ ]:
# ── 6. Unified normalization with class fallback ──────────────────────────────

MANUAL_ENTRIES = {
    'contrast': {
        'canonical_name': 'contrast media',
        'tty':            'manual',
        'level_1_class':  'Diagnostic agents',
        'level_1_id':     None,
        'level_2_class':  'Contrast media',
        'level_2_id':     None,
        'class_source':   'manual',
        'match_method':   'no_match'   # kept as requested
    }
}

def normalize_drug(drug_name, delay=0.15):
    time.sleep(delay)

    empty_row = {
        'input_drug':     drug_name,
        'matched_name':   None,
        'match_method':   'no_match',
        'rxcui':          None,
        'canonical_name': None,
        'tty':            None,
        'level_1_class':  None,
        'level_1_id':     None,
        'level_2_class':  None,
        'level_2_id':     None,
        'class_source':   None
    }

    # ── Manual entries ──
    key = clean_drug_name(drug_name).lower()
    if key in MANUAL_ENTRIES:
        return {**empty_row, 'matched_name': drug_name, **MANUAL_ENTRIES[key]}

    # ── Path A: treat as a specific drug via RxNorm ──
    rxcui, matched_name, method = get_rxcui(drug_name)

    if rxcui:
        props = get_concept_info(rxcui)
        hierarchy = get_atc_hierarchy(rxcui)
        snomed = get_snomed_from_rxcui(rxcui) if UMLS_API_KEY else None
        return {
            **empty_row,
            'matched_name':   matched_name,
            'match_method':   method,
            'rxcui':          rxcui,
            'canonical_name': props.get('canonical_name'),
            'tty':            props.get('tty'),
            'snomed_ct':      snomed,
            **hierarchy
        }

    # ── Path B: treat as a drug class via RxClass ──
    class_id, class_name, class_type, src = lookup_as_class(drug_name)

    if class_id:
        # Try to get the parent class one level up
        parent_id, parent_name = get_class_parent(class_id, source=src or 'ATC')
        return {
            **empty_row,
            'matched_name':   class_name,
            'match_method':   'rxclass_direct',
            'canonical_name': class_name,
            'tty':            class_type,        # e.g. 'EPC', 'MOA', 'CHEM'
            'level_1_class':  class_name,        # the entry itself is the class
            'level_1_id':     class_id,
            'level_2_class':  parent_name,       # one level above
            'level_2_id':     parent_id,
            'class_source':   src
        }

    # ── Path C: total failure ──
    return {**empty_row, 'match_method': 'no_match'}

In [ ]:
# ── 5. Main loop with checkpointing ─────────────────────────────────────────

def run_normalization(unique_drugs, save_every=50, delay=0.1):
    """
    Normalize all drugs, skipping already-completed ones.
    Saves to checkpoint every `save_every` new entries.
    """
    # Load any prior progress
    completed = load_checkpoint()          # {drug_name: result_dict}
    already_done = set(completed.keys())

    remaining = [d for d in unique_drugs if d not in already_done]
    print(f"{len(already_done)} already done, {len(remaining)} remaining.")

    new_since_save = 0

    for i, drug in enumerate(remaining):
        if i % 100 == 0:
            print(f"  {i}/{len(remaining)} remaining "
                  f"({len(completed)} total done)")

        result = normalize_drug(drug)
        completed[drug] = result
        new_since_save += 1

        # Periodic save
        if new_since_save >= save_every:
            save_checkpoint(completed)
            new_since_save = 0
            print(f"  Checkpoint saved ({len(completed)} entries)")

        time.sleep(delay)

    # Final save
    save_checkpoint(completed)
    print(f"Done. {len(completed)} total entries saved.")

    return list(completed.values())

In [ ]:
exploded = pd.read_csv('./data/MIMIC_IV_PATIENT_DRUG_ALLERGIES.csv')

In [ ]:
# ── 7. Run on exploded dataframe ─────────────────────────────────────────────

unique_drugs = exploded['drugs'].dropna().unique().tolist()
print(f"Normalizing {len(unique_drugs)} unique drug entries...")

records = run_normalization(unique_drugs, save_every=50, delay=0.1)

drug_lookup = pd.DataFrame(records)
result = exploded.merge(drug_lookup, left_on='drugs', right_on='input_drug', how='left')

In [ ]:
result

In [ ]:
result.to_csv('MIMIC_IV_ALLERGIEX_RXNORM.csv', index=False)

In [ ]:
print(result.iloc[174774][1])

In [ ]:
to_remove = ['ADRs on File', 'All allergies', 'adverse drug reactions previously recorded have been deleted', 'No Allergies']
to_remove_lower = {s.lower().strip() for s in to_remove}
result = result[~result['drugs'].str.lower().str.strip().isin(to_remove_lower)].reset_index(drop=True)
result

In [ ]:
to_merge = {
    'Iodine':       ['Iodine Containing', 'Iodine and Iodide Containing Products'],
    'Sulfonamides': ['Sulfa (Sulfonamide Antibiotics)']
}
merge_map = {variant: canonical 
             for canonical, variants in to_merge.items() 
             for variant in variants}
def apply_merge_map(name, merge_map):
    return merge_map.get(str(name).strip(), name)

result['drugs'] = result['drugs'].apply(lambda x: apply_merge_map(x, merge_map))
result = result.drop_duplicates(subset=['subject_id', 'drugs']).reset_index(drop=True)
result

In [ ]:
unnorm = result[result['matched_name'].isna()]
unnorm

In [ ]:
unique_unnorm = sorted(unnorm['drugs'].dropna().unique().tolist())
unique_unnorm

In [ ]:
unique_unnormed_with_idx = dict(enumerate(unique_unnorm))
import json
with open('unmatched_drugs.json', 'w') as f:
    json.dump(unique_unnormed_with_idx, f)

In [ ]:
import csv
with open('MIMIC_IV_DRUG_ALLERGY_non_drug_entries.csv', 'w', newline='') as f:
    writer = csv.writer(f)
    writer.writerow(['non_drug_entries'])
    writer.writerows([[entry] for entry in non_drug_entries])

In [ ]:
def remove_non_drug_entries(result_df, non_drug_entries):
    """
    Remove rows from result dataframe where drugs matches a non-drug entry.
    
    Parameters:
        result_df (pd.DataFrame): The result dataframe with a 'drugs' column
        non_drug_entries (list): List of non-drug strings to remove
    
    Returns:
        pd.DataFrame: Filtered dataframe with non-drug rows removed
    """
    # Normalise the lookup set: lowercase + strip whitespace/punctuation
    non_drug_set = {entry.strip().lower().strip('"* -') for entry in non_drug_entries}
    
    # Create a boolean mask: True = keep, False = drop
    mask = ~result_df['drugs'].str.strip().str.lower().str.strip('"* -').isin(non_drug_set)
    
    filtered_df = result_df[mask].reset_index(drop=True)
    
    n_removed = len(result_df) - len(filtered_df)
    print(f"Removed {n_removed} non-drug rows ({len(filtered_df)} remaining)")
    
    return filtered_df


# Usage
result = remove_non_drug_entries(result, non_drug_entries)

In [ ]:
result

In [ ]:
unnorm = remove_non_drug_entries(unnorm, non_drug_entries)
unnorm

In [ ]:
print(unnorm.loc[33104][1])

In [ ]:
to_merge = { 'Sulfonamides': ['Iodinated Contrast Media']}
merge_map = {variant: canonical 
             for canonical, variants in to_merge.items()
             for variant in variants}
def apply_merge_map(name, merge_map):
    return merge_map.get(str(name).strip(), name)

result['drugs'] = result['drugs'].apply(lambda x: apply_merge_map(x, merge_map))
unnorm['drugs'] = unnorm['drugs'].apply(lambda x: apply_merge_map(x, merge_map))

In [ ]:
unnorm

In [ ]:
unnorm['drugs'].to_csv('unmatched_drugs.csv', index=False)

In [ ]:
unique_unnorm = unnorm['drugs'].dropna().unique()
unique_unnorm

In [ ]:
unique_unnorm = pd.Series(unnorm['drugs'].dropna().unique())
unique_unnorm.to_csv('unmatched_drugs.csv', index=False)

In [ ]:
import json
# Local file, not in the repo: contains MIMIC-derived text. Do not commit.
with open('clinical_note_entries.json', encoding='utf-8') as f:
    clinical_note_entries = json.load(f)

In [ ]:

def remove_non_drug_entries(result_df, non_drug_entries):
    """
    Remove rows from result dataframe where drugs matches a non-drug entry.
    
    Parameters:
        result_df (pd.DataFrame): The result dataframe with a 'drugs' column
        non_drug_entries (list): List of non-drug strings to remove
    
    Returns:
        pd.DataFrame: Filtered dataframe with non-drug rows removed
    """
    # Normalise the lookup set: lowercase + strip whitespace/punctuation
    non_drug_set = {entry.strip().lower().strip('"* -') for entry in non_drug_entries}
    
    # Create a boolean mask: True = keep, False = drop
    mask = ~result_df['drugs'].str.strip().str.lower().str.strip('"* -').isin(non_drug_set)
    
    filtered_df = result_df[mask].reset_index(drop=True)
    
    n_removed = len(result_df) - len(filtered_df)
    print(f"Removed {n_removed} non-drug rows ({len(filtered_df)} remaining)")
    
    return filtered_df


# Usage
result = remove_non_drug_entries(result, clinical_note_entries)

In [ ]:
result

In [ ]:
unnorm = remove_non_drug_entries(unnorm, clinical_note_entries)
unnorm

output of this notebook:  a partially-normalized .csv MIMIC_IV_EXT_BHC_ALLERGYX_RXNORM_FULL.csv file. We'll deal with the rest separately because notebook too unwieldly now.